# 01 – Data Loading & Cleaning
Project: Olist E-commerce Analytics
Goal: inspect each raw table, document every issue found, fix what needs fixing, and save clean files to `data/processed/`.

Decision log convention: every finding ends in either an **Action** or "No action needed, because…".

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path('.').resolve().parent
RAW_DIR = BASE_DIR / 'data' / 'raw'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'

print(pd.__version__)
print(RAW_DIR.exists(), PROCESSED_DIR.exists())

3.0.6
True True


## Customers table
### Task 1: Load the raw file
Load `olist_customers_dataset.csv` into `customers_df` (raw, never modified) and work on `customers` (a copy).

In [3]:
customers_df = pd.read_csv(RAW_DIR / 'olist_customers_dataset.csv')
customers = customers_df.copy()
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


**Completed:** loaded the raw file into `customers_df`, working copy is `customers`.
**Result:** 5 columns: customer_id, customer_unique_id, customer_zip_code_prefix, customer_city, customer_state.

### Task 2: Structure and data types
Find the shape and the data type of every column.

In [4]:
print("customer table info:")
customers.info()
print("customer table shape:",customers.shape)


customer table info:
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB
customer table shape: (99441, 5)


**Completed:** checked shape and dtypes.
**Result:** 99,441 rows × 5 columns. Four columns are text, one (`customer_zip_code_prefix`) is int64.
**Finding:** the zip prefix should be text, not a number (handled in Task 8).

### Task 3: Missing values
Count nulls in every column.

In [5]:
customers.isnull().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

**Completed:** counted nulls per column.
**Result:** 0 nulls in all 5 columns.
**Action:** none needed, because the table is complete.

### Task 4: Duplicate rows
Check for fully duplicated rows.

In [6]:
print('Duplicate rows:', customers.duplicated().sum())

Duplicate rows: 0


**Completed:** checked duplicates.
**Result:** 0 duplicate rows.
**Action:** none needed, because there is nothing to remove.

### Task 5: `customer_id` vs `customer_unique_id`
Find out which column identifies a real person, and how many repeat customers exist.

In [7]:
n_ids = customers['customer_id'].nunique()
n_people = customers['customer_unique_id'].nunique()

print('customer_id unique:       ', n_ids)
print('customer_unique_id unique:', n_people)
print('Rows from returning customers:', n_ids - n_people) #repeat customers

# How many customer_id rows does each real person have?
customers['customer_unique_id'].value_counts().value_counts().sort_index()

customer_id unique:        99441
customer_unique_id unique: 96096
Rows from returning customers: 3345


count
1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: count, dtype: int64

**Completed:** compared the two ID columns.
**Result:** 99,441 `customer_id` values but only 96,096 `customer_unique_id` values, so 3,345 rows belong to returning customers.
**Finding:** `customer_id` is created per order; `customer_unique_id` identifies the real person.
**Action:** use `customer_unique_id` for every customer count, repeat-purchase and retention analysis in this project.

### Task 6: Customer distribution by state
Check state codes are valid and see where customers are concentrated.

In [8]:
state_counts = customers['customer_state'].value_counts()
state_share = (customers['customer_state'].value_counts(normalize=True) * 100).round(2)

print('Unique states:', len(state_counts))
print(pd.concat([state_counts, state_share], axis=1, keys=['customers', 'share_%']).head(5))
print('Top 3 share:', state_share.head(3).sum().round(1), '%')

Unique states: 27
                customers  share_%
customer_state                    
SP                  41746    41.98
RJ                  12852    12.92
MG                  11635    11.70
RS                   5466     5.50
PR                   5045     5.07
Top 3 share: 66.6 %


**Completed:** counted customers per state.
**Result:** 27 unique states (matches Brazil's 26 states + Federal District). SP 41,746 (42.0%), RJ 12,852 (12.9%), MG 11,635 (11.7%); top 3 states ≈ 66.6% of customers.
**Insight:** demand is heavily concentrated in the Southeast. Revisit this in the dashboard and in the delivery analysis.
**Action:** none needed.

### Task 7: Standardise city names
Check for whitespace, uppercase and accented characters that would split one city into several.

In [9]:
cities_before = customers['customer_city'].nunique()

city_clean = (customers['customer_city']
              .str.strip()
              .str.lower()
              .str.normalize('NFKD')
              .str.encode('ascii', errors='ignore')
              .str.decode('utf-8'))

print('Unique cities before:', cities_before)
print('Unique cities after: ', city_clean.nunique())
print('Rows changed:        ', (city_clean != customers['customer_city']).sum())

customers['customer_city'] = city_clean

Unique cities before: 4119
Unique cities after:  4119
Rows changed:         0


**Completed:** stripped spaces, lowercased and removed accents.
**Result:** unique cities stayed at 4,119 before and after; rows changed: <paste your number>.
**Finding:** city names were already standardised in the source.
**Action:** the step stays in the notebook as a safeguard, but no corrections were needed.

### Task 8: Fix the zip code prefix
Brazilian zip prefixes have 5 digits. Stored as integers, leading zeros are lost (9790 instead of 09790). Convert to a 5-character string.

In [10]:
customers['customer_zip_code_prefix'] = (customers['customer_zip_code_prefix']
                                         .astype(str)
                                         .str.zfill(5))

print(customers['customer_zip_code_prefix'].str.len().value_counts())
customers[['customer_zip_code_prefix']].head()

customer_zip_code_prefix
5    99441
Name: count, dtype: int64


,customer_zip_code_prefix
0,14409
1,09790
2,01151
3,08775
4,13056


**Completed:** converted the zip prefix to a zero-padded 5-character string.
**Result:** all 99,441 values have length 5; e.g. 9790 → 09790.
**Why it matters:** zip is an identifier, not a quantity, and it will be joined to the geolocation table later. Mismatched types or lost zeros would break that join.

### Task 9: Save the clean table
Save to `data/processed/customers_clean.csv`.
Note: when reading this file back, force the zip column to text with `dtype={'customer_zip_code_prefix': str}`, otherwise pandas turns it into an integer again.

In [11]:
customers.to_csv(PROCESSED_DIR / 'customers_clean.csv', index=False)

check = pd.read_csv(PROCESSED_DIR / 'customers_clean.csv',
                    dtype={'customer_zip_code_prefix': str})
print(check.shape)
print(check['customer_zip_code_prefix'].head(3).tolist())

(99441, 5)
['14409', '09790', '01151']


**Completed:** saved `customers_clean.csv` and verified it reloads with zip codes intact.

### Customers: decision log
| Finding | Action |
|---|---|
| No nulls, no duplicate rows | None needed |
| `customer_id` is per order (99,441); `customer_unique_id` is per person (96,096) | Use `customer_unique_id` for all customer-level analysis |
| 27 valid states; SP, RJ, MG ≈ 67% of customers | None; carry forward as a business insight |
| City names already clean | None needed (safeguard kept) |
| Zip prefix stored as integer, leading zeros lost | Converted to 5-character string |

# Table 2: orders

In [12]:
orders_df= pd.read_csv(RAW_DIR/"olist_orders_dataset.csv")
orders=orders_df.copy()
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [14]:
print("orders shape:",orders.shape)
print("orders info:",orders.info())


orders shape: (99441, 8)
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB
orders info: None


In [19]:
print("orders null value:\n",orders.isnull().sum())
print("duplicate orders:",orders.duplicated().sum())

orders null value:
 order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
duplicate orders: 0


In [25]:
orders.nunique()

order_id                         99441
customer_id                      99441
order_status                         8
order_purchase_timestamp         98875
order_approved_at                90733
order_delivered_carrier_date     81018
order_delivered_customer_date    95664
order_estimated_delivery_date      459
dtype: int64